[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/33_beam_search.ipynb)

# 🟡 Medium: Beam Search Decoding

*Inference & Decoding*
Implement **beam search** over a black-box scoring function, ranked by
length-normalised log-probability.

```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token,
                length_penalty=1.0):
    ...  # -> list[int], the best sequence, starting with start_token
```

`log_prob_fn(tokens)` receives the sequence decoded so far (including
`start_token`) and returns a `(V,)` array of log-probabilities for the **next**
token. Read `V` off that first call — do not assume it.

### The algorithm, exactly
Start with one live hypothesis `(start_token,)` at score `0.0`. Repeat at most
`max_len - 1` times:

1. **Expand.** For every live beam $b$ and every token $v$, the candidate score
   is $s_b + \log p_b(v)$.
2. **Terminate.** Every candidate whose new token is `eos_token` is finished:
   move it to the finished set. It is never extended again.
3. **Prune.** The best `beam_width` *non-eos* candidates become the new live
   set. If none survive, stop.

Anything still live when the loop ends is also a (truncated) hypothesis. Return
the finished hypothesis maximising

$$\mathrm{score}_{\text{norm}} = \frac{\sum_t \log p(y_t \mid y_{<t})}{L^{\alpha}}, \qquad L = \text{tokens generated after } \texttt{start\_token}$$

with $\alpha =$ `length_penalty`. $\alpha = 0$ recovers the raw sum;
$\alpha = 1$ is the mean log-probability per token.

### Rules
- No Python sort over `n_live * V` candidate tuples — build the `(n_live, V)`
  score matrix and take `jax.lax.top_k` on its flattened view
- Prune on **raw** sums, rank the final answer on **normalised** scores
- `beam_width = 1` must reduce to greedy decoding
- Return a plain Python `list[int]` beginning with `start_token`
- `len(result) <= max_len`

### Why length normalisation is not optional
Every $\log p$ is negative, so appending a token can only ever *lower* a raw
sum. Raw-score beam search therefore has a structural bias toward stopping
early: it prefers a two-token answer at $-0.4$ over a five-token answer at
$-0.9$ even though the latter is a much more confident model of its own tokens
($-0.225$ vs $-0.4$ per token). Un-normalised beam search in machine
translation empirically truncates long sentences; GNMT's fix is
$((5+L)/6)^\alpha$, a smoothed version of the $L^\alpha$ used here.

### Why you should usually not use it anyway
Beam search answers "what is the most likely continuation?" — and for
open-ended generation that is the wrong question. The mode of a language model
is bland: high-likelihood text is generic, repetitive, and degenerate
("I don't know. I don't know. I don't know."), because real human text is not
the argmax of its own distribution — it sits in a band of moderate surprisal.
Widening the beam makes this *worse*, not better: you search harder for a mode
you did not want.

So beam search belongs where the output is genuinely constrained and near-unique
— translation, speech recognition, constrained code or JSON generation,
anything scored by BLEU/WER — and top-k/top-p sampling belongs in chat and
open-ended writing. The other cost is systems-level: a width-$B$ beam is $B$
concurrent KV caches and $B\times$ the attention memory, and every step needs a
global top-k across beams, which is a synchronisation point that pure sampling
never pays for.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import math

import jax
import jax.numpy as jnp


def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token,
                length_penalty=1.0):
    """Decode the best sequence under log_prob_fn with a beam of beam_width.

    Args:
        log_prob_fn:       callable, tokens-so-far -> (V,) next-token log-probs
        start_token:    int, the first token of every hypothesis
        max_len:        int, maximum total sequence length (incl. start_token)
        beam_width:     int, number of live hypotheses kept per step
        eos_token:      int, token that terminates a hypothesis
        length_penalty: float alpha in score / L**alpha, L = generated length

    Returns:
        list[int] — the best sequence, starting with start_token.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax.numpy as jnp

# A toy model: token 1 is a safe-but-mediocre step, token 2 is a gamble that
# pays off, token 3 is eos.
def log_prob_fn(tokens):
    lp = jnp.full((4,), -100.0)
    if len(tokens) == 1:
        lp = lp.at[1].set(-0.5).at[2].set(-0.9)
    elif tokens[-1] == 1:
        lp = lp.at[3].set(-2.5)
    elif tokens[-1] == 2:
        lp = lp.at[3].set(-0.1)
    else:
        lp = lp.at[3].set(-50.0)
    return lp


print("greedy   (width 1):", beam_search(log_prob_fn, 0, 4, 1, 3))
print("beam     (width 2):", beam_search(log_prob_fn, 0, 4, 2, 3))
print("-> greedy commits to the locally better token 1 and pays for it later")


# Length normalisation flips the winner.
def steady(tokens):
    lp = jnp.full((4,), -20.0)
    if len(tokens) - 1 < 3:
        lp = lp.at[1].set(-0.3).at[3].set(-0.4)
    else:
        lp = lp.at[3].set(0.0)
    return lp


print("alpha=0 (raw sums):", beam_search(steady, 0, 5, 2, 3, length_penalty=0.0))
print("alpha=1 (per token):", beam_search(steady, 0, 5, 2, 3, length_penalty=1.0))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("beam_search")

# hint("beam_search")      # stuck? nudge without the answer
# solution("beam_search")  # spoiler: the reference implementation
# status()                 # your dashboard across all problems